# Predictive maintenance — medallion pipeline

End-to-end notebook following the **Bronze → Silver** medallion (Gold later).
Per source: **Bronze** (raw, operators pseudonymised) then **Silver** (treated),
each with the same per-feature understanding (reused from `src.common.profiling`,
identical to the `dataset_report.md`). Full automated run:
`uv run python scripts/run_pipeline.py`.

## 0. Setup

In [ ]:
import sys, tempfile
from pathlib import Path

%load_ext autoreload
%autoreload 2

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

%matplotlib inline

from src import config
from src.common.env import load_dotenv
from src.common.profiling import (
    feature_synthesis_markdown,
    plot_feature_by_machine,
    plot_feature_distribution,
)
load_dotenv(PROJECT_ROOT / '.env')

def show_per_feature(df, numeric_features, machine_col=config.MACHINE_COLUMN):
    out = Path(tempfile.mkdtemp(prefix='nb_'))
    for col in df.columns:
        display(Markdown(f'#### {col}'))
        display(Markdown(feature_synthesis_markdown(df, col)))
        if col in numeric_features:
            i = numeric_features.index(col) + 1
            if machine_col in df.columns:
                display(Image(filename=str(plot_feature_by_machine(df, col, i, out, machine_col))))
            display(Image(filename=str(plot_feature_distribution(df, col, i, out))))

## 1. Sources — Bronze & Silver

For each source we build the **Bronze** DataFrame (raw, via `load_bronze`) then the
**Silver** DataFrame (treated, via `to_silver`), and show the per-feature
understanding of each — same content as the reports.

### Incidents

In [ ]:
from src.sources.incidents.runner import load_bronze, to_silver, BRONZE_NUMERIC, SILVER_NUMERIC

incidents_bronze = load_bronze()
incidents_silver = to_silver(incidents_bronze)
print('bronze', incidents_bronze.shape, '| silver', incidents_silver.shape)

#### Incidents — Bronze (raw, per-feature)

In [ ]:
show_per_feature(incidents_bronze, BRONZE_NUMERIC)

#### Incidents — Silver (treated, per-feature)

In [ ]:
show_per_feature(incidents_silver, SILVER_NUMERIC)

### Telemetry

In [ ]:
from src.sources.telemetry.runner import load_bronze, to_silver, BRONZE_NUMERIC, SILVER_NUMERIC

telemetry_bronze = load_bronze()
telemetry_silver = to_silver(telemetry_bronze)
print('bronze', telemetry_bronze.shape, '| silver', telemetry_silver.shape)

#### Telemetry — Bronze (raw, per-feature)

In [ ]:
show_per_feature(telemetry_bronze, BRONZE_NUMERIC)

#### Telemetry — Silver (treated, per-feature)

In [ ]:
show_per_feature(telemetry_silver, SILVER_NUMERIC)

### Machines

In [ ]:
from src.sources.machines.runner import load_bronze, to_silver, BRONZE_NUMERIC, SILVER_NUMERIC

machines_bronze = load_bronze()
machines_silver = to_silver(machines_bronze)
print('bronze', machines_bronze.shape, '| silver', machines_silver.shape)

#### Machines — Bronze (raw, per-feature)

In [ ]:
show_per_feature(machines_bronze, BRONZE_NUMERIC)

#### Machines — Silver (treated, per-feature)

In [ ]:
show_per_feature(machines_silver, SILVER_NUMERIC)

## 2. Database (bronze / silver schemas)

The orchestrator (`scripts/run_pipeline.py`) loads each layer into its schema.
Here we read the tables back. Start the DB with `docker compose up -d` if needed.

In [ ]:
from sqlalchemy import text
from src.database.engine import is_available, get_engine

if is_available():
    engine = get_engine()
    with engine.connect() as conn:
        for schema in ('bronze', 'silver'):
            for table in ('incidents', 'telemetry', 'maintenance'):
                n = conn.execute(text(f'SELECT count(*) FROM {schema}."{table}"')).scalar()
                print(f'{schema}.{table:12} {n} rows')
else:
    print('PostgreSQL unavailable — run: docker compose up -d, then scripts/run_pipeline.py')

## 3. Cross-source analysis

**A. via `src/`** (joined graphs) and **B. inline** (machine-profile correlation).

In [ ]:
from src.analyses.joins import load_sources, build_machine_profile, build_reactive_incident_join
from src.analyses import plots as cross_plots

sources = load_sources()
profile = build_machine_profile(sources)
reactive = build_reactive_incident_join(sources)
out = Path(tempfile.mkdtemp(prefix='nb_cross_'))
for png in cross_plots.plot_all(profile, reactive, out):
    display(Image(filename=str(png)))

In [ ]:
num = profile.select_dtypes('number')
num.corr().round(2)

---
Reminder: this notebook is for exploration. The reproducible pipeline is
`scripts/run_pipeline.py` (re-run after any data update in `data/raw/`).